# Classifier Calibration

Reliability of the class probabilities of the failure-mode classifiers on the random test split (expected calibration error and accuracy against confidence).

## Imports

In [ ]:
import sys
import pickle
import warnings
import numpy as np
import pandas as pd
import joblib

sys.path.append('../src')
from zoo import TASKS, load_task

warnings.filterwarnings('ignore')

## Settings

In [ ]:
RESULTS = '../results'
zoo = pd.concat([pd.read_csv(f'{RESULTS}/{t}__zoo_results.csv') for t in TASKS], ignore_index=True)

## Classification Tasks: Reliability

In [ ]:
reliability = {}
for task in [t for t in TASKS if t.endswith('FailureMode')]:
    df, X, y, groups, is_clf, target = load_task(task)
    predictions = pickle.load(open(f'{RESULTS}/{task}__predictions.pkl', 'rb'))
    y_test = predictions[zoo[zoo['File'] == task]['Model'].iloc[0]]['y_test']
    rel = {}
    for name in zoo[zoo['File'] == task]['Model']:
        proba = predictions[name]['proba']
        if proba is None:
            continue
        model = joblib.load(f'{RESULTS}/models/{task}__{name}.joblib')
        pred = model.classes_[proba.argmax(1)]
        conf = proba.max(1)
        correct = (pred == y_test).astype(float)
        bins = np.linspace(0, 1, 6)
        bin_id = np.clip(np.digitize(conf, bins) - 1, 0, 4)
        bx = [conf[bin_id == k].mean() if (bin_id == k).any() else np.nan for k in range(5)]
        by = [correct[bin_id == k].mean() if (bin_id == k).any() else np.nan for k in range(5)]
        bn = [(bin_id == k).sum() for k in range(5)]
        ece = float(sum(n * abs(a - b) for n, a, b in zip(bn, bx, by) if n > 0) / len(conf))
        order = np.argsort(-conf)
        retention = np.cumsum(correct[order]) / np.arange(1, len(order) + 1)
        rel[name] = dict(ece=ece, bx=bx, by=by, bn=bn, retention=retention, acc=float(correct.mean()), conf_mean=float(conf.mean()))
    reliability[task] = rel
pd.DataFrame({t: {m: v['ece'] for m, v in r.items()} for t, r in reliability.items()}).round(3)

## Save

In [ ]:
with open(f'{RESULTS}/reliability.pkl', 'wb') as fh:
    pickle.dump(reliability, fh)